# SlidyReplay (playbook v0.2.0)

Generates MP4 videos of sliding puzzle replays.

In [ ]:
#@title ## Setup
import os, subprocess, sys

def run(cmd, show=False):
    r = subprocess.run(cmd, shell=True, capture_output=not show, text=True)
    if r.returncode != 0 and r.stderr:
        sys.stderr.write(r.stderr)
    return r.returncode == 0

if not os.path.exists("slidyreplay"):
    ok = run("git clone https://github.com/dphdmn/slidyreplay.git")
    print(f"{'  OK' if ok else 'FAIL'} git clone")
if os.path.basename(os.getcwd()) != "slidyreplay":
    os.chdir("slidyreplay")
ok = run("pip install -q -r requirements.txt")
print(f"{'  OK' if ok else 'FAIL'} pip install")
if ok:
    run("apt-get install -y -qq software-properties-common")
    run("add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg6")
    ok = run("apt-get update -qq") and run("apt-get install -y -qq ffmpeg")
    print(f"{'  OK' if ok else 'FAIL'} ffmpeg install")
    if ok:
        run("ffmpeg -version 2>&1 | head -2", show=True)
import torch
has_cuda = torch.cuda.is_available()
print(f"{'  OK' if has_cuda else 'FAIL'} CUDA" + (f" ({torch.cuda.get_device_name(0)})" if has_cuda else " NOT AVAILABLE"))

In [ ]:
#@title ## Generate Replay
import ipywidgets as widgets
from IPython.display import display, Video, clear_output
import subprocess, sys, re, os, tempfile, glob
from concurrent.futures import ThreadPoolExecutor

# ─── Mode selector ───
mode = widgets.Dropdown(options=['URL', 'Custom', 'File', 'Batch'], value='URL',
                        description='Mode:', style={'description_width': 'initial'})

url = widgets.Text(
    value='https://slidysim.github.io/replay?r=eNqVVNmOm0gU%2FRWLV6wGzN7qZIRN2wY3GLPY2EkUFVRhMPtuSPLvg7sz0czjqB6q7jl3qSrpnC8%2FsD6CTYg9c3MsRNE1bN6PCQIQVV4OKmgPBcKesSZKETbH%2FDxrqjypJ0TL2%2FoBXUGKtBw%2BkqwGZHCqmVAY1UUCht%2FV%2FyKyKX0dJQ2qJhwWIUyzCQXdNUEZ9kzNPyY986IozrE079A0a0HRJElOVPEIOG4hfKTVDUiLqYinBZ6hKXaxoKYGdZ506DsEDfgOOhAlwEumhk3VojnWRRDlSZTF0%2FCwmfo9E0Tf909D3jath578PCV60PjhX92n%2B4nbHOhR3bnxdMOoPiEPew5AUqNfcwypdz15bS1WPmvminjlLBk3DfNkcaTjWO2Wp3EdQfnsWbhjmrKqbrkap%2FVgxBErGnwe8TCkWXGrBfa433c82fpBNZJuk3AG0a6lNDBo1icYWXUIb9m7Wdset%2F7uZhBLjuVXYmhsETEkiOEZhItjd%2FMJAwbCG0NwKm6wrhiwAkVBOuC6c4NfXK7LcPUo8jyOE2JZoDyFarZzvUQs0KYqScBmQLt1byAiZFhnCboAWkDOjhEaf%2BdtrQCeFkefpc2bdxI8ao%2BjCpYMHTS6Ww%2FCeekbeutaqnlXVkq6l%2FqzQNAHaRt6tHVg76wqFYdYibJV7L4tolaOd%2FvDyOo6s2dOr7q0Xh2LzXK1lx0nPFpHLSr3yKs3blHWC58qcS6xu6o0bPfyRku1nNUsujmlGZjgMKYjJEPL5vXyAgTd66E1OscNeXh9UzbBWjO1XI03TXHkpXKhl45b3948LVywMrysVktqMJUIX68ZMlzuap%2FvPHkAJgfrUmPs27W%2FUxW6ddVg24lIjZZrmMF%2BXx79Klfd7VFje8%2FRHVko%2B4Lt7rUEs2J%2FWpJr7nVnZ%2BXmoMk7fOPUw8m%2BX3EjeCtiF17129omr5f8oPV%2BWErWIdbl1t45pKacR3kV9%2FGViiRDkfxzdo2vN1Jpr5vidXCE%2BKwV5Vm1Pak%2BKa8MM%2BaiEql92nMbmeIMt9ABbYB4bETVovxRryWVVgYbh2axFC%2BWLUvVWtlaq9Yu%2FbYI4FnZrnjDF%2BVqI1%2BFW06im6QyaHB4SQOa75%2BrThGtojycW8odFS3PL0TWq50xMHa83tEqTBxjn66nP1D2K%2FkoL9ISmiGZ1tpQjdtdq5ZFEVHCPRZg0IsL0a0PdYXNf0xysqu2bhD8UOav%2BYesuWlhL3UBspmfgLr%2B9BUDSRGCKYr8%2BKHkr9jnlyi9%2FqFRmt%2Bir9isrvwpilJwRTXxj5wjP58okDQTdc5be8Jm7xYw%2B%2BMMj37%2FmXcyjeUEfjmZ32YvxIP6bD0cZeYNsw%2B%2FevGqz%2FZkPzNjOfs5e%2FfBaV%2BQC%2FaJXDyR7O%2BqyTT%2Bn8W8vxSbP%2F7j29%2F7Pdq8',
    description='URL:', layout=widgets.Layout(width='100%'))

# ─── Custom mode ───
solution = widgets.Text(value='RU2RDLULD2RU2RDL3DRULUR3D2L2U2L', description='Solution:')
size = widgets.Text(value='4x4', description='Size:')
tps = widgets.FloatText(value=26.960, description='TPS:')
time_ = widgets.FloatText(value=0, description='Time (s):')
scramble = widgets.Text(value='1 2 3 4/9 10 0 7/11 8 15 6/5 14 13 12', description='Scramble:')
movetimes = widgets.Text(value='0,60,98,120,158,203,256,308,330,359,390,444,471,496,547,586,637,660,719,766,810,832,861,976,976,976,998,1020,1043,1095,1110,1178,1224', description='Movetimes (csv):', layout=widgets.Layout(width='100%'))

custom_fields = widgets.VBox([
    solution, size,
    widgets.HBox([tps, time_]),
    scramble, movetimes
])

# ─── File mode ───
file_content = widgets.Textarea(value='', description='Paste URL or solution:',
                                layout=widgets.Layout(width='100%', height='80px'))
file_upload_btn = widgets.Button(description='Upload File', button_style='info')
file_info = widgets.Label(value='')

# ─── Batch mode ───
batch_text = widgets.Textarea(value='', description='URLs / solutions (one per line):',
                              layout=widgets.Layout(width='100%', height='150px'))
batch_upload_btn = widgets.Button(description='Upload Batch File', button_style='info')
batch_info = widgets.Label(value='')

# ─── Common settings ───
fps = widgets.IntSlider(value=60, min=5, max=240, step=5, description='FPS:')
quality = widgets.FloatSlider(value=1.0, min=1.0, max=4.0, step=0.5, description='Quality:')
compression = widgets.IntSlider(value=18, min=10, max=40, step=1, description='CRF:')
output = widgets.Text(value='replay.mp4', description='Output:')

# ─── Advanced settings ───
speedup = widgets.FloatText(value=1.0, description='Speed (x):', step=0.1)
force_fringe = widgets.Checkbox(value=False, description='Force fringe')
no_gpu = widgets.Checkbox(value=False, description='Disable GPU')
log = widgets.Checkbox(value=False, description='Enable logging')

no_layout = widgets.Checkbox(value=False, description='Grid only (no layout)')
no_border = widgets.Checkbox(value=False, description='No tile borders')
no_secondary_border = widgets.Checkbox(value=False, description='No sec. borders')
no_numbers = widgets.Checkbox(value=False, description='No tile numbers')

advanced = widgets.VBox([
    speedup,
    widgets.HBox([force_fringe, no_gpu, log]),
    no_layout,
    widgets.HBox([no_border, no_secondary_border, no_numbers]),
])
accordion = widgets.Accordion(children=[advanced])
accordion.set_title(0, 'Advanced settings')

# ─── Visibility toggles ───
def toggle_mode(change):
    v = change['new'] if isinstance(change, dict) else change.new
    url.layout.display = 'flex' if v == 'URL' else 'none'
    custom_fields.layout.display = 'flex' if v == 'Custom' else 'none'
    file_content.layout.display = 'flex' if v == 'File' else 'none'
    file_upload_btn.layout.display = 'flex' if v == 'File' else 'none'
    file_info.layout.display = 'flex' if v == 'File' else 'none'
    batch_text.layout.display = 'flex' if v == 'Batch' else 'none'
    batch_upload_btn.layout.display = 'flex' if v == 'Batch' else 'none'
    batch_info.layout.display = 'flex' if v == 'Batch' else 'none'
    output.layout.display = 'flex' if v != 'Batch' else 'none'

mode.observe(toggle_mode, names='value')
toggle_mode({'new': mode.value})

# ─── File upload handler ───
def on_file_upload(b):
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn, content in uploaded.items():
            data = content.decode('utf-8') if isinstance(content, bytes) else content
            file_content.value = data.strip()
            file_info.value = f'Loaded: {fn} ({len(data)} chars)'
    except ImportError:
        file_info.value = 'File upload only available in Google Colab.'
file_upload_btn.on_click(on_file_upload)

def on_batch_upload(b):
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn, content in uploaded.items():
            data = content.decode('utf-8') if isinstance(content, bytes) else content
            batch_text.value = data.strip()
            lines = [l for l in data.strip().splitlines() if l.strip() and not l.strip().startswith('#')]
            batch_info.value = f'Loaded: {fn} ({len(data)} chars, {len(lines)} entries)'
    except ImportError:
        batch_info.value = 'File upload only available in Google Colab.'
batch_upload_btn.on_click(on_batch_upload)

# ─── Buttons ───
btn = widgets.Button(description='Generate', button_style='primary', layout=widgets.Layout(width='200px'))
dl_btn = widgets.Button(description='\u2b07 Download', disabled=True, layout=widgets.Layout(width='200px'))
out = widgets.Output()

def _run(args):
    from IPython.display import clear_output
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    noise_re = re.compile(r'(ffmpeg version|built with|configuration:|libav|libswscale|libswresample|libpostproc|Input #|Stream mapping:|Duration:|Stream #|Output #|\[libx264|\[mp4 @|\[out#0|\[h264_nvenc|frame= |Lsize=|Metadata:|encoder :|Side data|cpb:)')
    kept = []
    prog = [None]
    import threading
    lock = threading.Lock()
    def redraw():
        clear_output(wait=True)
        lines = list(kept)
        if prog[0] is not None:
            lines.append(prog[0])
        sys.stdout.write('\n'.join(lines) + ('\n' if lines else ''))
        sys.stdout.flush()
    def read_stream(stream, is_err):
        for line in iter(stream.readline, ''):
            if noise_re.search(line):
                continue
            with lock:
                if 'Rendering frames:' in line:
                    prog[0] = line.rstrip('\n')
                else:
                    if prog[0] is not None:
                        kept.append(prog[0]); prog[0] = None
                    kept.append(line.rstrip('\n'))
                redraw()
    with ThreadPoolExecutor(max_workers=2) as pool:
        pool.submit(read_stream, p.stdout, False)
        pool.submit(read_stream, p.stderr, True)
    p.wait()
    with lock:
        if prog[0] is not None:
            kept.append(prog[0]); prog[0] = None
        redraw()
    return p.returncode

def on_generate(b):
    with out:
        clear_output(wait=True)
        print('Generating replay, please wait...', flush=True)

        base = ['python', 'main.py', '--fps', str(fps.value),
                '--quality', str(quality.value), '--compression', str(compression.value)]
        if speedup.value != 1.0:
            base.extend(['--speedup', str(speedup.value)])
        if force_fringe.value:
            base.append('--force-fringe')
        if no_gpu.value:
            base.append('--no-gpu')
        if log.value:
            base.append('--log')
        if no_layout.value:
            base.append('--no-layout')
        if no_border.value:
            base.append('--no-border')
        if no_secondary_border.value:
            base.append('--no-secondary-border')
        if no_numbers.value:
            base.append('--no-numbers')

        mv = mode.value
        tmp_files = []

        try:
            if mv == 'URL':
                args = base + ['--url', url.value, '-o', output.value]
                ret = _run(args)
            elif mv == 'Custom':
                args = base + ['--solution', solution.value, '-o', output.value]
                if size.value.strip():
                    args.extend(['--size', size.value.strip()])
                if tps.value > 0:
                    args.extend(['--tps', str(tps.value)])
                if time_.value > 0:
                    args.extend(['--time', str(time_.value)])
                if scramble.value.strip():
                    args.extend(['--scramble', scramble.value.strip()])
                if movetimes.value.strip():
                    args.extend(['--movetimes', movetimes.value.strip()])
                ret = _run(args)
            elif mv == 'File':
                tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
                tmp.write(file_content.value)
                tmp.close()
                tmp_files.append(tmp.name)
                args = base + ['--file', tmp.name, '-o', output.value]
                ret = _run(args)
            else:
                tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
                tmp.write(batch_text.value)
                tmp.close()
                tmp_files.append(tmp.name)
                args = base + ['--batch', tmp.name, '-o', 'batch_output.mp4']
                ret = _run(args)
        finally:
            for p in tmp_files:
                if os.path.exists(p):
                    os.unlink(p)

        if mv == 'Batch':
            seen = set()
            for f in sorted(glob.glob('batch_output*.mp4')):
                if f not in seen:
                    seen.add(f)
                    sz = os.path.getsize(f) / 1e6
                    print(f"\n{'='*50}")
                    print(f'File: {f}  ({sz:.2f} MB)')
                    print(f"{'='*50}")
                    display(Video(f, embed=True, width=640))
            dl_btn.disabled = False
        elif os.path.exists(output.value):
            sz = os.path.getsize(output.value) / 1e6
            print(f"\n{'='*50}")
            print(f'File: {output.value}  ({sz:.2f} MB)')
            print(f"{'='*50}")
            display(Video(output.value, embed=True, width=640))
            dl_btn.disabled = False

def on_download(b):
    from google.colab import files
    files.download(output.value)

btn.on_click(on_generate)
dl_btn.on_click(on_download)

display(mode, url, custom_fields,
        widgets.VBox([file_upload_btn, file_content, file_info]),
        widgets.VBox([batch_upload_btn, batch_text, batch_info]),
        widgets.HBox([fps, quality, compression]),
        output, accordion,
        widgets.HBox([btn, dl_btn]), out)


In [ ]:
#@title ## Run Benchmarks

benchmark_mode = "All" #@param ["All", "Small (CPU+GPU)", "Big (GPU only)"]

if benchmark_mode == "All":
    !python benchmark.py
elif benchmark_mode == "Small (CPU+GPU)":
    !python benchmark.py --small
else:
    !python benchmark.py --big